# 02 - Classical Baseline: TF-IDF + Logistic Regression
**Author:** Member 2 (Ravindu Sehan)

Benchmark linear classification on CFPB complaints against Deep Learning models.

In [ ]:
import os
import sys
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

sys.path.append('..')
from src.data import load_splits
from src.evaluate import evaluate_model, time_inference, save_results

SEED = 42
np.random.seed(SEED)

In [ ]:
# Load data splits
train, val, test, meta = load_splits('../data')
print(f'Train samples: {len(train.text):,}')
print(f'Val samples:   {len(val.text):,}')
print(f'Classes:       {meta["classes"]}')

In [ ]:
# Feature extraction using TF-IDF (train only)
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english', sublinear_tf=True)
t0 = time.time()
X_train_tfidf = tfidf.fit_transform(train.text)
X_val_tfidf = tfidf.transform(val.text)
print(f'TF-IDF fit in {time.time() - t0:.2f}s | Shape: {X_train_tfidf.shape}')

In [ ]:
# Balanced Logistic Regression
clf = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, solver='lbfgs', random_state=SEED)
t0 = time.time()
clf.fit(X_train_tfidf, train.y)
train_time_s = time.time() - t0
print(f'Trained in {train_time_s:.2f}s')

In [ ]:
# Validation Evaluation
y_val_pred = clf.predict(X_val_tfidf)
y_val_proba = clf.predict_proba(X_val_tfidf)

res_baseline = evaluate_model(val.y, y_val_pred, y_val_proba, meta['classes'], split='val')
res_baseline['params'] = int((X_train_tfidf.shape[1] * meta['n_classes']) + meta['n_classes'])
res_baseline['train_time_s'] = float(train_time_s)

def predict_fn(batch_text):
    return clf.predict_proba(tfidf.transform(batch_text))

res_baseline['inference_ms_per_1k'] = time_inference(predict_fn, val.text[:1000])

save_results('baseline_tfidf', res_baseline, config={'C': 1.0, 'max_features': 10000, 'ngram_range': [1, 2]}, seed=SEED)
print('Baseline Macro F1:', res_baseline.get('f1_macro', res_baseline.get('macro_f1')))